# Online Shoppers — Revenue Prediction
**Goal:** Predict whether a visitor will make a purchase (`Revenue = 1`) or not (`Revenue = 0`).  
**Key feature:** `PageValues` (how valuable the pages visited were).  
**Pipeline:** load → normalize → compare models → pick the best one.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (
    f1_score, roc_auc_score, roc_curve,
    classification_report, ConfusionMatrixDisplay
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

sns.set_theme(style='whitegrid', palette='muted')
print('Libraries loaded.')

## 2. Load Raw Data

In [ ]:
URL = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00468/online_shoppers_intention.csv'
df_raw = pd.read_csv(URL)
print('Shape:', df_raw.shape)
df_raw.head(3)

## 3. Exploratory Data Analysis — PageValues vs Revenue

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# --- Histogram ---
for label, color in [(False, 'steelblue'), (True, 'tomato')]:
    axes[0].hist(
        df_raw[df_raw['Revenue'] == label]['PageValues'],
        bins=50, alpha=0.65,
        label='Bought' if label else 'Not bought',
        color=color
    )
axes[0].set_xlabel('PageValues')
axes[0].set_title('PageValues distribution by Revenue')
axes[0].legend()

# --- Box plot ---
df_raw.boxplot(column='PageValues', by='Revenue', ax=axes[1])
axes[1].set_title('PageValues: buyers vs non-buyers')
axes[1].set_xlabel('Revenue')
plt.suptitle('')

# --- Correlation bar ---
df_enc = df_raw.copy()
df_enc['Revenue'] = df_enc['Revenue'].astype(int)
df_enc['Weekend'] = df_enc['Weekend'].astype(int)
df_enc = pd.get_dummies(df_enc, columns=['Month', 'VisitorType'], drop_first=True)
num = df_enc.select_dtypes(include='number')
corr = num.corr()['Revenue'].drop('Revenue').sort_values(key=abs, ascending=False).head(10)
corr.plot(kind='barh', ax=axes[2],
          color=['tomato' if v > 0 else 'steelblue' for v in corr])
axes[2].axvline(0, color='black', lw=0.8)
axes[2].set_title('Top 10 correlations with Revenue')
axes[2].set_xlabel('Pearson r')

plt.tight_layout()
plt.show()

print('Mean PageValues — not bought:', df_raw[df_raw['Revenue']==False]['PageValues'].mean().round(2))
print('Mean PageValues — bought:    ', df_raw[df_raw['Revenue']==True ]['PageValues'].mean().round(2))

## 4. Preprocessing & Normalization

In [ ]:
df = df_raw.copy()

# --- Encode booleans ---
df['Revenue'] = df['Revenue'].astype(int)
df['Weekend'] = df['Weekend'].astype(int)

# --- One-hot encode categoricals ---
df = pd.get_dummies(df, columns=['Month', 'VisitorType'], drop_first=True)

# --- Separate target ---
y = df['Revenue']
X = df.drop('Revenue', axis=1)

# --- Normalize ALL features with StandardScaler ---
# (PageValues and other numeric columns are skewed; scaling stabilises models)
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

print('Features after preprocessing:', X_scaled.shape[1])
print('Target balance:')
print(y.value_counts().rename({0: 'Not bought', 1: 'Bought'}))
print(f'\nPageValues — mean after scaling: {X_scaled["PageValues"].mean():.4f}, std: {X_scaled["PageValues"].std():.4f}')

## 5. Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape[0]}  |  Test: {X_test.shape[0]}')
print(f'Positive rate — Train: {y_train.mean():.3f}  |  Test: {y_test.mean():.3f}')

## 6. Model Comparison — 5-Fold Cross-Validation

In [ ]:
models = {
    'Decision Tree':       DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=100, random_state=42),
    'SVM':                 SVC(probability=True, random_state=42),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}

print(f'{'Model':<24}  CV F1 mean ± std')
print('-' * 46)
for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='f1')
    cv_results[name] = scores
    print(f'{name:<24}  {scores.mean():.4f} ± {scores.std():.4f}')

## 7. Test-Set Evaluation

In [ ]:
test_results = []
fitted_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    fitted_models[name] = model
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    test_results.append({
        'Model':         name,
        'CV F1 (mean)':  cv_results[name].mean(),
        'Test F1':       f1_score(y_test, y_pred),
        'ROC AUC':       roc_auc_score(y_test, y_prob),
    })

results_df = (
    pd.DataFrame(test_results)
    .sort_values('ROC AUC', ascending=False)
    .reset_index(drop=True)
)
results_df.index += 1
results_df.style.background_gradient(subset=['ROC AUC', 'Test F1'], cmap='Greens')

## 8. ROC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
colors = ['steelblue', 'forestgreen', 'tomato', 'darkorange', 'purple']

for (name, model), color in zip(fitted_models.items(), colors):
    y_prob = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    ax.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', color=color, lw=2)

ax.plot([0, 1], [0, 1], 'k--', lw=1)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — All Models')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Best Model — Detailed Report

In [ ]:
best_row   = results_df.iloc[0]
best_name  = best_row['Model']
best_model = fitted_models[best_name]

print('=' * 55)
print(f'  BEST MODEL: {best_name}')
print('=' * 55)
print(f'  CV F1  : {best_row["CV F1 (mean)"]:.4f}')
print(f'  Test F1: {best_row["Test F1"]:.4f}')
print(f'  ROC AUC: {best_row["ROC AUC"]:.4f}')
print('=' * 55)

y_pred_best = best_model.predict(X_test)
print(f'\nClassification report ({best_name}):')
print(classification_report(y_test, y_pred_best, target_names=['Not bought', 'Bought']))

### Confusion Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_best,
    display_labels=['Not bought', 'Bought'],
    cmap='Blues', ax=ax
)
ax.set_title(f'Confusion Matrix — {best_name}')
plt.tight_layout()
plt.show()

### Feature Importances (if available)

In [ ]:
if hasattr(best_model, 'feature_importances_'):
    feat_imp = (
        pd.Series(best_model.feature_importances_, index=X_scaled.columns)
        .sort_values(ascending=False)
        .head(12)
    )
    # Highlight PageValues bar
    colors_imp = ['tomato' if f == 'PageValues' else 'teal' for f in feat_imp.index]
    plt.figure(figsize=(9, 5))
    feat_imp.plot(kind='barh', color=colors_imp)
    plt.title(f'Top 12 Feature Importances — {best_name}')
    plt.xlabel('Importance')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()
elif hasattr(best_model, 'coef_'):
    coef = pd.Series(best_model.coef_[0], index=X_scaled.columns)
    coef_top = coef.abs().sort_values(ascending=False).head(12)
    colors_coef = ['tomato' if f == 'PageValues' else 'steelblue' for f in coef_top.index]
    plt.figure(figsize=(9, 5))
    coef_top.plot(kind='barh', color=colors_coef)
    plt.title(f'Top 12 Feature Coefficients (abs) — {best_name}')
    plt.xlabel('|Coefficient|')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

## 10. Summary

| Step | Detail |
|------|--------|
| Dataset | UCI Online Shoppers Intention (12 330 sessions) |
| Target | `Revenue` — 0 = did not buy, 1 = bought (≈15 % positive) |
| Normalization | `StandardScaler` on all features after one-hot encoding |
| Key feature | `PageValues` — strongest single predictor of purchase |
| Selection metric | ROC AUC (handles class imbalance better than accuracy) |
| Best model | determined above by ROC AUC on the held-out test set |